#Init

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

#Reading crm_sales_details from Bronze

In [0]:
df = spark.table("workspace.bronze.crm_sales_details_raw")

#Transforming the data

## Changing column names

In [0]:
COLUMN_RENAME = {
    "sls_ord_num" : "order_number",
    "sls_prd_key" : "product_key",
    "sls_cust_id" : "customer_id",
    "sls_order_dt" : "order_date",
    "sls_ship_dt" : "shipping_date",
    "sls_due_dt" : "due_date",
    "sls_sales" : "sales",
    "sls_quantity" : "quantity",
    "sls_price" : "price"
}

for old_name, new_name in COLUMN_RENAME.items():
    df = df.withColumnRenamed(old_name, new_name)

## Trimming the data

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

## Converting dt into date format yyyy-MM-dd

### Validating date before transformation

#### Checking for length of each date should be exact 8

In [0]:
for field in ["order_date", "shipping_date", "due_date"]:
    bad_data = df.filter(length(col(f"{field}").cast("string")) != 8)
    bad_count = bad_data.count()
    print(f"Bad data found for {field} is {bad_count}")
    if bad_count > 0:
        bad_data.groupBy(field).count().show()
        

#### Checking count of date field having null values

In [0]:
for field in ["order_date", "shipping_date", "due_date"]:
    null_count = df.filter(col(f"{field}").isNull()).count()
    print(f"Null count for {field} is {null_count}")

#### Checking date format. Middle part should be month <= 12 and end part should be date <= 31

In [0]:
for field in ["order_date", "shipping_date", "due_date"]:
    valid_df = df.filter(length(col(field)) == 8)
    string_date = col(field).cast("string")
    month_val = string_date.substr(5, 2).cast("int")
    day_val = string_date.substr(7, 2).cast("int")

    bad_month_count = valid_df.filter((month_val < 1) | (month_val > 12)).count()
    bad_day_count = valid_df.filter((day_val < 1) | (day_val > 31)).count()

    print(f"Bad month count for {field} is {bad_month_count}")
    print(f"Bad day count for {field} is {bad_day_count}")
        

In [0]:
from pyspark.sql.types import IntegerType

# Idempotency guard: skip if the date columns are already converted,
# so accidentally running this cell twice does not corrupt df
if not isinstance(df.schema["order_date"].dataType, IntegerType):
    print("Date columns already converted — skipping this cell.")
else:
    ship_date = try_to_date(col("shipping_date").cast("string"), "yyyyMMdd")

    df = df.withColumns({
        # Invalid order_date (e.g. 0) -> derive from shipping_date - 7 days
        "order_date": when(length(col("order_date").cast("string")) == 8,
                           to_date(col("order_date").cast("string"), "yyyyMMdd"))
                      .otherwise(date_sub(ship_date, 7)),
        # Invalid shipping/due dates -> NULL (no reliable derivation rule)
        "shipping_date": ship_date,
        "due_date": try_to_date(col("due_date").cast("string"), "yyyyMMdd"),
    })
    df.printSchema()


### Replace incorrect date column with null.

In [0]:
for field in ['order_date', 'shipping_date', 'due_date']:
    check_df = df.filter(col(field).isNull())
    print(f"Null date for {field} is {check_df.count()}")

# Show the actual rows with an invalid (nulled) order_date
df.filter(col("order_date").isNull()).show(truncate=False)


In [0]:
df.limit(10).display()

## Writing data to Silver table

In [0]:
df.write\
.mode("overwrite")\
.format("delta")\
.saveAsTable("workspace.silver.crm_sales_info")